In [71]:
from google.colab import userdata
import os

HUGGING_FACE_TOKEN = userdata.get('HUGGING_FACE_TOKEN')

In [72]:
!pip install openai

In [73]:
import os
from openai import OpenAI


client = OpenAI(
   base_url="https://router.huggingface.co/v1",
   api_key=HUGGING_FACE_TOKEN,
)


response = client.chat.completions.create(
   model="openai/gpt-oss-120b:cerebras",
   #model="mistralai/Mistral-7B-Instruct-v0.2:featherless-ai",
   messages=[{"role": "user", "content": "Tell me a fun fact about the Eiffel Tower."}],
)


print(response.choices[0].message.content)

A fun fact about the Eiffel Tower: when it was first built in 1889, it was painted a shade of **Venetian red**—very different from the sleek “Eiffel Tower Brown” you see today. In fact, the tower has been repainted roughly once every seven years, and each repaint requires about 60 tons of paint and around 1,700 liters of primer! This ongoing makeover helps protect its iron structure from rust and keeps the iconic silhouette shining for generations of visitors.


In [74]:
!pip install sentence_transformers

In [75]:
import numpy as np
from sentence_transformers import SentenceTransformer

In [76]:
documentos = [
   "Una lista en Python es una colección ordenada y mutable.",
   "Para añadir elementos a una lista se usa el método .append().",
   "Las tuplas son inmutables, no se pueden modificar."
]

In [77]:
model = SentenceTransformer("paraphrase-MiniLM-L3-v2", device="cpu")

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-MiniLM-L3-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [78]:
model.encode(documentos[0])

array([-0.43020314, -0.2527027 ,  0.04236107, -0.23067226,  0.05062285,
        0.18194015,  0.24300355, -0.12273794,  0.21523051,  0.07221638,
        0.30657887, -0.09708156,  0.00245   ,  0.11848677,  0.5463478 ,
       -0.2446167 ,  0.44182205, -0.17636438,  0.0770957 , -0.22872818,
        0.03937786,  0.21128489,  0.15698445, -0.01758418,  0.1704435 ,
        0.13133468, -0.11322768, -0.13088912, -0.04056824, -0.08780703,
       -0.17368978,  0.05000307,  0.14309144, -0.02601322, -0.32086784,
        0.02640766,  0.14418456, -0.20323706,  0.00446206,  0.35812047,
        0.14334539,  0.32944942,  0.0053365 ,  0.08655484,  0.16674583,
       -0.11172453,  0.1845551 , -0.2500967 , -0.07795476, -0.08686865,
        0.4974594 , -0.01536229, -0.2467701 , -0.05435563, -0.16176498,
        0.11089483,  0.1033567 ,  0.02129018, -0.33227202, -0.34195468,
       -0.49045432,  0.0151961 , -0.08650155,  0.08097621, -0.05182578,
        0.3006295 ,  0.04001084, -0.21948649,  0.16643827,  0.05

In [79]:
embeddings_docs = [model.encode(doc) for doc in documentos]
print(embeddings_docs)

[array([-0.43020314, -0.2527027 ,  0.04236107, -0.23067226,  0.05062285,
        0.18194015,  0.24300355, -0.12273794,  0.21523051,  0.07221638,
        0.30657887, -0.09708156,  0.00245   ,  0.11848677,  0.5463478 ,
       -0.2446167 ,  0.44182205, -0.17636438,  0.0770957 , -0.22872818,
        0.03937786,  0.21128489,  0.15698445, -0.01758418,  0.1704435 ,
        0.13133468, -0.11322768, -0.13088912, -0.04056824, -0.08780703,
       -0.17368978,  0.05000307,  0.14309144, -0.02601322, -0.32086784,
        0.02640766,  0.14418456, -0.20323706,  0.00446206,  0.35812047,
        0.14334539,  0.32944942,  0.0053365 ,  0.08655484,  0.16674583,
       -0.11172453,  0.1845551 , -0.2500967 , -0.07795476, -0.08686865,
        0.4974594 , -0.01536229, -0.2467701 , -0.05435563, -0.16176498,
        0.11089483,  0.1033567 ,  0.02129018, -0.33227202, -0.34195468,
       -0.49045432,  0.0151961 , -0.08650155,  0.08097621, -0.05182578,
        0.3006295 ,  0.04001084, -0.21948649,  0.16643827,  0.0

In [80]:
def cosine_similarity(a, b):
   return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [81]:
# Primera Similitud

pregunta = "lista en python"
emb_pregunta = model.encode(pregunta)
similitud_1 = cosine_similarity(emb_pregunta, embeddings_docs[0])
similitud_1

np.float32(0.6333943)

In [88]:
def get_context(question):
    # 1. Generar el embedding de la pregunta
    emb_question = model.encode(question)

    # 2. Calcular similitudes
    similarities = [cosine_similarity(emb_question, doc) for doc in embeddings_docs]

    # 3. Filtrar documentos que superen el umbral (0.5)
    # Usamos zip para iterar sobre documentos y similitudes al mismo tiempo
    context_list = [doc for doc, sim in zip(documentos, similarities) if sim > 0.5]

    print(f"Similitudes encontradas: {similarities}")

    # 4. Retornar los documentos unidos en un solo string para el prompt
    if not context_list:
        return "No se encontró información relevante en los documentos."

    return "\n".join(context_list)

In [89]:
get_context(pregunta)

Similitudes encontradas: [np.float32(0.6333943), np.float32(0.46258584), np.float32(0.13697082)]


'Una lista en Python es una colección ordenada y mutable.'

In [101]:
def ask(question):
    context = get_context(question)

    prompt = f"""
    Actúa como un tutor de Python. Tu tarea es responder preguntas basadas exclusivamente en los documentos proporcionados.

    Instrucciones:
    1. Usa solo el contexto de abajo.
    2. Si la respuesta no se encuentra en el texto, di: "Lo siento, la información solicitada no está disponible en mi base de conocimientos."
    3. Mantén la respuesta breve y educativa.

    ### Contexto:
    {context}

    ### Pregunta del usuario:
    {question}

    ### Respuesta sugerida:
    """

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b:cerebras",
        # model="mistralai/Mistral-7B-Instruct-v0.2:featherless-ai",
        messages=[{"role": "user", "content": prompt}],
    )

    return response.choices[0].message.content

In [102]:
print(get_context("¿Qué es una lista?"))

Similitudes encontradas: [np.float32(0.6148077), np.float32(0.51833844), np.float32(0.21575356)]
Una lista en Python es una colección ordenada y mutable.
Para añadir elementos a una lista se usa el método .append().


In [103]:
print(ask("¿Qué es una lista?"))

Similitudes encontradas: [np.float32(0.6148077), np.float32(0.51833844), np.float32(0.21575356)]
Una lista en Python es una colección ordenada y mutable.


In [104]:
print(ask("¿Una lista solo permite enteros?"))

Similitudes encontradas: [np.float32(0.38218194), np.float32(0.53366834), np.float32(0.1612018)]
Lo siento, la información solicitada no está disponible en mi base de conocimientos.
